#  Clustering con k-means: tipos de oleaje en Bilbao

**Analítica de datos aplicada a la Ingeniería Civil**

**Caso:** boya de Bilbao-Vizcaya (Puertos del Estado) · unos 59 000 estados de mar entre 1990 y 2009 · altura significante (Hs), periodo de pico (Tp) y dirección (Dir)

> 🎯 **Mensaje clave:** k-means encuentra **tipos de oleaje** reconocibles sin saber nada de meteorología, pero preparar bien las variables y elegir el número de grupos es cosa del ingeniero.

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
- Las celdas marcadas con **✏️** tienen un hueco `___` que debes rellenar. Si no lo rellenas, la celda dará error.
- Los bloques **💡 Preguntas** son para pensar y comentar en clase: lo importante es **interpretar**, no escribir código.

| Bloque | Qué hacemos en este notebook |
|:--:|---|
| 1 | Un ejemplo sencillo en 2D |
| 2 | Los datos de la boya |
| 3 | Preparar la matriz: escalas y dirección circular |
| 4 | ¿Cuántos grupos? Codo y silueta |
| 5 | Interpretar los grupos y ver cuándo aparecen |
| 6 | Clasificar un estado de mar nuevo |
| 7 | *Ampliación (si sobra tiempo):* 50 estados de mar representativos |

### Imports y configuración básica

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Colores que usaremos en todo el notebook
AZUL = "#4C72B0"
NARANJA = "#DD8452"
ROJO = "#C44E52"
COLORES_GRUPOS = plt.cm.tab10.colors   # un color por grupo de k-means
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

plt.rcParams["font.size"] = 12


def estilo(ax):
    """Estética común de las gráficas: sin bordes arriba y a la derecha, rejilla suave."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)

---
## 1. Un ejemplo sencillo en 2D

### Formulación

Dados $N$ puntos $\mathbf{x}_i$ y $k$ centroides iniciales al azar, k-means repite dos pasos hasta que los grupos no cambian:

1. **Asignar** cada punto al centroide más cercano: $\; j(i) = \arg\min_j \lVert \mathbf{x}_i - \mathbf{v}_j \rVert$
2. **Actualizar** cada centroide como la media de su grupo: $\; \mathbf{v}_j = \dfrac{1}{n_j} \sum_{\mathbf{x}_i \in C_j} \mathbf{x}_i$

Así se minimiza la **inercia** $J = \sum_j \sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \mathbf{v}_j \rVert^2$. Como el resultado depende del arranque, se repite varias veces (`n_init`) y se guarda la solución de menor $J$.

✏️ **Rellena el hueco:** elige el número de grupos (empieza por `5` y después prueba `2`).

In [ ]:
x = [1, 4, 5, 8, 15, 18, 50, 4, 7, 8, 9, 3, 30]
y = [6, 8, 9, 5, 1, 2, 3, 8, 9, 10, 14, 17, 14]
matriz_2d = np.vstack([x, y]).T          # puntos x dimensiones

n_grupos = ___   # ✏️
km2d = KMeans(n_clusters=n_grupos, n_init=10, random_state=0).fit(matriz_2d)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, c=[COLORES_GRUPOS[g] for g in km2d.labels_], s=80, alpha=0.6)
ax.scatter(km2d.cluster_centers_[:, 0], km2d.cluster_centers_[:, 1], s=220, marker="X",
           c=COLORES_GRUPOS[:n_grupos], edgecolor="black", label="Centroides")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(); estilo(ax); plt.show()

> ### 💡 Preguntas
>
> 1) ¿Qué le pasa al punto (50, 3) con k = 5? ¿Y con k = 2?
>
> 2) ¿Qué nos dice eso sobre k-means y los valores alejados del resto?

---
## 2. Los datos de la boya

El fichero tiene una matriz con una fila por estado de mar. Nos quedamos con la fecha, **Hs** (m), **Tp** (s) y **Dir** (grados de procedencia: 0° = norte, 90° = este, 270° = oeste).

In [ ]:
# Si trabajas en local, cambia la ruta por la de tu ordenador
ruta = "/Users/beatrizperezdiaz/Library/CloudStorage/OneDrive-UNICAN/DOCENCIA/2026_2027/AnaliticaDeDatos_PersonalBPD/MaterialBea_2aParte/Codes/AnaliticaDeDatos-main/datos/clases/clase_3_pca_clustering/Bilbao-Vizcaya_Ext.mat"
array = loadmat(ruta)["matriz_15_Boya"]

datos = pd.DataFrame({"year": array[:, 0], "month": array[:, 1], "day": array[:, 2], "hour": array[:, 3],
                      "hs": array[:, 4], "tp": array[:, 6], "dir": array[:, 9]})
datos["fecha"] = pd.to_datetime(datos[["year", "month", "day", "hour"]])
datos = datos.drop(columns=["year", "day", "hour"]).set_index("fecha")

print(len(datos), "estados de mar entre", datos.index[0].date(), "y", datos.index[-1].date())
datos[["hs", "tp", "dir"]].describe().round(1)

Una función para dibujar las tres parejas de variables (la usaremos dos veces):

In [ ]:
def dibuja_parejas(grupos=None, centros=None):
    """Hs-Tp, Dir-Tp y Dir-Hs. Si se dan grupos, colorea cada estado de mar por su grupo."""
    color = AZUL if grupos is None else [COLORES_GRUPOS[g] for g in grupos]
    pares = [("hs", "tp", "Hs (m)", "Tp (s)"), ("dir", "tp", "Dir (°)", "Tp (s)"), ("dir", "hs", "Dir (°)", "Hs (m)")]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, (vx, vy, lx, ly) in zip(axes, pares):
        ax.scatter(datos[vx], datos[vy], c=color, s=2, alpha=0.3)
        if centros is not None:
            ax.scatter(centros[vx], centros[vy], s=220, marker="X", c=COLORES_GRUPOS[:len(centros)],
                       edgecolor="black", zorder=3)
        ax.set_xlabel(lx); ax.set_ylabel(ly); estilo(ax)
    plt.tight_layout(); plt.show()

dibuja_parejas()

> ### 💡 Preguntas
>
> 1) ¿De qué dirección llega la mayor parte del oleaje? ¿Y los estados de mar más altos?
>
> 2) ¿Se ven grupos claramente separados a simple vista?

---
## 3. Preparar la matriz: escalas y dirección circular

K-means mide **distancias**, así que hay que cuidar dos cosas:

- **Hs y Tp** tienen unidades y rangos distintos: los llevamos a [0, 1]:

$$\tilde x = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- **La dirección es circular** (359° y 1° son casi la misma). La pasamos a **seno y coseno**: cada dirección es un punto de un círculo. Las multiplicamos por 0,5 para que su rango también sea 1:

$$\text{Dir} \;\rightarrow\; \left(0{,}5\,\sin\theta,\; 0{,}5\,\cos\theta\right)$$

✏️ **Rellena el hueco:** la cuarta columna es el **coseno** de la dirección en radianes (`rad`).

In [ ]:
minis = datos[["hs", "tp"]].min()
maxis = datos[["hs", "tp"]].max()


def prepara(hs, tp, direccion):
    """Matriz para k-means: Hs y Tp en [0, 1] y la dirección como seno y coseno."""
    rad = np.deg2rad(direccion)
    return np.column_stack([(hs - minis["hs"]) / (maxis["hs"] - minis["hs"]),
                            (tp - minis["tp"]) / (maxis["tp"] - minis["tp"]),
                            0.5 * np.sin(rad),
                            0.5 * ___])   # ✏️


X = prepara(datos["hs"].values, datos["tp"].values, datos["dir"].values)
print("Matriz para k-means:", X.shape)

> ### 💡 Preguntas
>
> 1) ¿Por qué no usamos la dirección tal cual, en grados?
>
> 2) ¿Qué pasaría si no lleváramos Hs y Tp a [0, 1]?

---
## 4. ¿Cuántos grupos? Codo y silueta

- **Codo:** la inercia $J$ siempre baja al añadir grupos. Buscamos el $k$ a partir del cual añadir otro grupo apenas la reduce.
- **Silueta:** para cada estado de mar, $a$ = distancia media a su grupo y $b$ = al grupo vecino:

$$s = \frac{b - a}{\max(a, b)} \qquad (-1 \le s \le 1)$$

Una media por encima de 0,5 indica grupos claros; por debajo de 0,25, estructura débil. La calculamos con una muestra de 5000 estados de mar para que sea rápida.

✏️ **Rellena el hueco:** dentro del bucle, el número de grupos es `k`.

In [ ]:
ks = range(2, 11)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=___, n_init=10, random_state=0).fit(X)   # ✏️
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X, km.labels_, sample_size=5000, random_state=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inercias, "o-", color=AZUL, lw=2); axes[0].set_title("Método del codo")
axes[0].set_ylabel("Inercia J")
axes[1].plot(ks, siluetas, "o-", color=ROJO, lw=2); axes[1].set_title("Silueta media")
axes[1].axhspan(0, 0.25, color="grey", alpha=0.1)
for ax in axes:
    ax.set_xlabel("Número de grupos k"); estilo(ax)
plt.tight_layout(); plt.show()

> ### 💡 Preguntas
>
> 1) ¿Hay un codo claro?
>
> 2) ¿Qué k sugiere la silueta? ¿Te fiarías solo de ella?

---
## 5. Interpretar los grupos

Para leer los centroides en **unidades físicas** deshacemos la preparación: Hs y Tp con los mismos mínimos y máximos, y la dirección con $\theta = \operatorname{atan2}(\sin\theta, \cos\theta)$.

✏️ **Rellena el hueco:** elige `k` (proponemos `4`; después prueba `6`).

In [ ]:
k = ___   # ✏️
km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
datos["grupo"] = km.labels_

c = km.cluster_centers_
centros = pd.DataFrame({
    "hs": c[:, 0] * (maxis["hs"] - minis["hs"]) + minis["hs"],
    "tp": c[:, 1] * (maxis["tp"] - minis["tp"]) + minis["tp"],
    "dir": np.rad2deg(np.arctan2(c[:, 2], c[:, 3])) % 360,
    "% del tiempo": datos["grupo"].value_counts(normalize=True).sort_index() * 100})
centros.index.name = "grupo"
centros.round(1)

In [ ]:
dibuja_parejas(grupos=datos["grupo"], centros=centros)

> ### 💡 Preguntas
>
> 1) Pon nombre a cada grupo (por ejemplo: «temporal del noroeste», «mar de viento del nordeste»…).
>
> 2) ¿Qué grupo te preocuparía más para diseñar un dique? ¿Y para la operatividad de un puerto?

### ¿Cuándo aparece cada tipo de oleaje?

In [ ]:
tabla = pd.crosstab(datos["month"], datos["grupo"], normalize="index") * 100

fig, ax = plt.subplots(figsize=(10, 4))
abajo = np.zeros(12)
for g in range(k):
    ax.bar(range(12), tabla[g].values, bottom=abajo, color=COLORES_GRUPOS[g], width=0.85, label=f"Grupo {g}")
    abajo += tabla[g].values
ax.set_xticks(range(12), MESES); ax.set_ylabel("% de los estados de mar del mes")
ax.legend(bbox_to_anchor=(1.01, 1)); estilo(ax); plt.show()

> ### 💡 Preguntas
>
> 1) ¿En qué meses dominan los temporales? ¿Qué pasa en verano?

---
## 6. Clasificar un estado de mar nuevo

Un estado de mar nuevo se prepara **con los mismos mínimos y máximos** que los datos con los que entrenamos y se asigna al centroide más cercano (`predict`).

✏️ **Rellena el hueco:** escribe dos estados de mar `[Hs, Tp, Dir]`: un temporal `[7, 15, 290]` y un oleaje corto del nordeste `[1, 6, 40]`.

In [ ]:
nuevos = np.array(___)   # ✏️
grupo_nuevo = km.predict(prepara(nuevos[:, 0], nuevos[:, 1], nuevos[:, 2]))

for estado, g in zip(nuevos, grupo_nuevo):
    print(f"Hs = {estado[0]} m, Tp = {estado[1]} s, Dir = {estado[2]}°  →  grupo {g}")

> ### 💡 Preguntas
>
> 1) ¿A qué tipo de oleaje pertenece cada uno? ¿Tiene sentido?
>
> 2) Hs = 7 m está muy por encima del centroide de su grupo (≈ 2,9 m). ¿Es un problema?

---
## 7. *Ampliación (si sobra tiempo):* 50 estados de mar representativos

En ingeniería de costas se usa k-means con muchos grupos para elegir **estados de mar representativos**: en lugar de propagar con un modelo numérico los 59 000 estados de mar hasta la costa, se propagan solo los centroides y el resto se interpola.

In [ ]:
km50 = KMeans(n_clusters=50, n_init=10, random_state=0).fit(X)
c50 = km50.cluster_centers_
centros50 = pd.DataFrame({
    "hs": c50[:, 0] * (maxis["hs"] - minis["hs"]) + minis["hs"],
    "tp": c50[:, 1] * (maxis["tp"] - minis["tp"]) + minis["tp"],
    "dir": np.rad2deg(np.arctan2(c50[:, 2], c50[:, 3])) % 360})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (vx, vy, lx, ly) in zip(axes, [("hs", "tp", "Hs (m)", "Tp (s)"), ("dir", "hs", "Dir (°)", "Hs (m)")]):
    ax.scatter(datos[vx], datos[vy], s=2, alpha=0.2, color="lightgrey")
    ax.scatter(centros50[vx], centros50[vy], s=60, color=ROJO, edgecolor="black", zorder=3)
    ax.set_xlabel(lx); ax.set_ylabel(ly); estilo(ax)
plt.tight_layout(); plt.show()

> ### 💡 Pregunta
>
> 1) ¿Cubren los 50 centroides bien los temporales más fuertes (Hs > 8 m)? ¿Por qué?

---
### 📌 Para llevarse

- **K-means alterna dos pasos:** asignar cada punto al centroide más cercano y recalcular los centroides como medias. Minimiza la inercia.
- **Preparar las variables es parte del método:** escalas comparables y la dirección como seno y coseno.
- **El codo y la silueta orientan, pero no deciden:** con estructura débil, $k$ se elige porque los grupos se interpretan bien.
- **Los centroides se leen en unidades físicas:** tipos de oleaje con su Hs, Tp, dirección y frecuencia.
- **Un centroide es una media:** sirve para describir y clasificar, pero no para los extremos.